In [15]:
import pandas as pd

# Define the path to the saved Excel file
file_path = r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx"

# Read the Excel file into a DataFrame
df = pd.read_excel(file_path)

# Display the first five rows
display(df.head())

# Check the DataFrame structure
df.info()

,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.0
1,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert,1 000 t Getreidewert,10.0
2,1991/92,Gerste,Detailposition,Industrielle Verwertung für Energie,1 000 t Produktgewicht,NaN
3,1991/92,Gerste,Detailposition,Saatgut,1 000 t Produktgewicht,388.0
4,1991/92,Gerste,Detailposition,Industrielle Verwertung,1 000 t Produktgewicht,2671.0


<class 'pandas.DataFrame'>
RangeIndex: 1734 entries, 0 to 1733
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Wirtschaftsjahr     1734 non-null   str    
 1   Bilanz Produkt      1734 non-null   str    
 2   Bilanz Oberbegriff  1734 non-null   str    
 3   Bilanz Position     1734 non-null   str    
 4   Bilanz Einheit      1734 non-null   str    
 5   Bilanz Wert         1670 non-null   float64
dtypes: float64(1), str(5)
memory usage: 81.4 KB


# Import- und Exportquoten berechnen

### Official FAO Import Dependency Ratio (IDR)

The Food and Agriculture Organization of the United Nations (FAO) defines the Import Dependency Ratio (IDR) as the share of imports in the domestic supply, calculated as follows:

IDR = Imports / Production + Imports - Exports X 100

According to the FAO, this indicator measures the extent to which a country's domestic food supply depends on imports. The calculation assumes that imports are primarily used for domestic utilization rather than re-exported.

**Source:**

Food and Agriculture Organization of the United Nations (FAO). (2001). *Food Balance Sheets: A Handbook*. Chapter IV, Section 2: "Import Dependency Ratio (IDR)".

https://www.fao.org/4/X9892E/X9892E04.html


## Methodological Background: Supply Balances and Production

According to the German Federal Ministry of Agriculture (BMEL), the self-sufficiency rate is calculated using domestic production and utilization. The BMEL also explains that domestic utilization is determined while taking foreign trade into account.

**Sources:**
- BMEL Statistics: [Fundamentals of Supply Balances](https://www.bmel-statistik.de/ernaehrung/versorgungsbilanzen/grundlagen)
- BMEL Statistics: [Cereal Supply Balances](https://www.bmel-statistik.de/ernaehrung/versorgungsbilanzen/getreide)

These sources confirm that production, imports, exports, and domestic utilization are distinct components of a supply balance. However, they do not explicitly establish that *total production* (*Gesamterzeugung*) and *usable production* (*verwendbare Erzeugung*) are identical.

### Usable Production in the Cereal Supply Balance

The metadata for the cereal supply balance dataset published by the Federal Office for Agriculture and Food (BLE) explicitly state:

> "Usable production = harvest" (*Verwendbare Erzeugung = Ernte*).

**Source:**
- BLE Open Data: [Cereal Supply Balance — Dataset and Metadata](https://open-data.ble.de/dataset/versorgungsbilanz-getreide)

This clarifies the meaning of usable production in the cereal supply balance. However, the metadata do not explicitly confirm that usable production is identical to the separate variable *total production*.

### Implications for the Import and Export Ratios

**Export ratio (own defined indicator NOT FAO definition)**

If the objective is to measure the share of the harvest that is exported, the following formula is a reasonable and transparent indicator:

Export Ratio % = Exports / Production + Imports - Exports X 100


### Conclusion

The BMEL's description of the self-sufficiency rate as a ratio of domestic production to utilization supports the general approach of deriving indicators from supply balance data. However, it does not automatically validate every alternative ratio formula.

For this analysis, the import and export ratios are therefore treated as explicitly defined descriptive indicators. Their formulas, denominators, and interpretations are stated transparently, and usable production is used as the measure of the harvest in the cereal supply balance.


**Wichtiger Hinweis für später:** Die FAO definiert keine eigenständige, allgemein standardisierte Exportquote in ihrem Handbuch Food Balance Sheets: A Handbook (2001). Sie definiert jedoch den Self-Sufficiency Ratio (SSR) und den Import Dependency Ratio (IDR). 

A Handbook

Für deine wissenschaftliche Arbeit ist daher wichtig, die beiden Kennzahlen nicht zu verwechseln:

- FAO Self-Sufficiency Ratio (SSR)/Selbstversorgungsgrad:

SSR = Production / Production + Imports − Exports × 100

# Hier: Production = Verwendbare Erzeugung

Export hier: im Verhältnis zur rechnerischen Verfügbarkeit (Nenner wie Importquote)

In [16]:
import pandas as pd
import numpy as np

# 1. Reshape the data from long format to wide format
# Each row represents one financial year and one product
# Each balance position becomes a separate column
df_wide = df.pivot(
    index=["Wirtschaftsjahr", "Bilanz Produkt"],
    columns="Bilanz Position",
    values="Bilanz Wert"
).reset_index()

# Remove the column index name
df_wide.columns.name = None

# 2. Create a DataFrame for the calculated quotas
# Keep the financial year and product as identifiers
quota_results = df_wide[
    ["Wirtschaftsjahr", "Bilanz Produkt"]
].copy()

# 3. Calculate the import quota as a percentage
# Formula: Imports / (Usable production + Imports - Exports) * 100
quota_results["Importquote"] = (
    df_wide["Einfuhr"]
    / (
        df_wide["Verwendbare Erzeugung"]
        + df_wide["Einfuhr"]
        - df_wide["Ausfuhr"]
    )
    * 100
)

# 4. Calculate the export quota as a percentage
# Formula: Exports / (Usable production + Imports - Exports) * 100
quota_results["Exportquote"] = (
    df_wide["Ausfuhr"]
    / (
        df_wide["Verwendbare Erzeugung"]
        + df_wide["Einfuhr"]
        - df_wide["Ausfuhr"]
    )
    * 100
)

# 5. Replace infinite values with missing values
# This handles division by zero
quota_results[["Importquote", "Exportquote"]] = (
    quota_results[["Importquote", "Exportquote"]]
    .replace([np.inf, -np.inf], np.nan)
)

# 6. Sort the results by product and financial year
quota_results = quota_results.sort_values(
    ["Bilanz Produkt", "Wirtschaftsjahr"]
)

# 7. Display the calculated quotas rounded to two decimal places
display(
    quota_results.round({
        "Importquote": 2,
        "Exportquote": 2
    })
)

# 8. Check the number of observations and missing values
print("Number of observations:", len(quota_results))
print("Missing import quotas:", quota_results["Importquote"].isna().sum())
print("Missing export quotas:", quota_results["Exportquote"].isna().sum())

,Wirtschaftsjahr,Bilanz Produkt,Importquote,Exportquote
0,1991/92,Gerste,10.11,29.02
3,1992/93,Gerste,9.85,25.94
6,1993/94,Gerste,13.05,37.48
9,1994/95,Gerste,14.02,37.27
12,1995/96,Gerste,16.28,38.44
...,...,...,...,...
89,2020/21,Weichweizen,29.63,58.28
92,2021/22,Weichweizen,30.88,51.56
95,2022/23,Weichweizen,29.26,49.95
98,2023/24,Weichweizen,39.02,57.38


Number of observations: 102
Missing import quotas: 0
Missing export quotas: 0


In [17]:
import numpy as np
import pandas as pd

# 1. Recalculate the denominator using the original formulas
denominator_check = (
    df_wide["Verwendbare Erzeugung"]
    + df_wide["Einfuhr"]
    - df_wide["Ausfuhr"]
)

# 2. Recalculate the import quota independently
import_check = (
    df_wide["Einfuhr"]
    / denominator_check
    * 100
).replace([np.inf, -np.inf], np.nan)

# 3. Recalculate the export quota independently
export_check = (
    df_wide["Ausfuhr"]
    / denominator_check
    * 100
).replace([np.inf, -np.inf], np.nan)

# 4. Compare the recalculated values with the existing results
# Differences should be zero, apart from negligible floating-point deviations
import_diff = (
    quota_results["Importquote"] - import_check
).abs()

export_diff = (
    quota_results["Exportquote"] - export_check
).abs()

# 5. Display the verification results
print("Maximum import quota difference:", import_diff.max())
print("Maximum export quota difference:", export_diff.max())

print(
    "Import quotas mathematically consistent:",
    import_diff.fillna(0).lt(1e-9).all()
    and import_diff.isna().equals(import_check.isna())
)

print(
    "Export quotas mathematically consistent:",
    export_diff.fillna(0).lt(1e-9).all()
    and export_diff.isna().equals(export_check.isna())
)

Maximum import quota difference: 0.0
Maximum export quota difference: 0.0
Import quotas mathematically consistent: True
Export quotas mathematically consistent: True


In [18]:
# Check for missing input values
input_columns = [
    "Verwendbare Erzeugung",
    "Einfuhr",
    "Ausfuhr"
]

print("Missing input values:")
print(df_wide[input_columns].isna().sum())

# Calculate the denominator
denominator = (
    df_wide["Verwendbare Erzeugung"]
    + df_wide["Einfuhr"]
    - df_wide["Ausfuhr"]
)

# Check for zero and negative denominators
print("Zero denominators:", denominator.eq(0).sum())
print("Negative denominators:", denominator.lt(0).sum())

# Display any observations with non-positive denominators
display(
    df_wide.loc[
        denominator.le(0),
        [
            "Wirtschaftsjahr",
            "Bilanz Produkt",
            *input_columns
        ]
    ]
)

Missing input values:
Verwendbare Erzeugung    0
Einfuhr                  0
Ausfuhr                  0
dtype: int64
Zero denominators: 0
Negative denominators: 0


,Wirtschaftsjahr,Bilanz Produkt,Verwendbare Erzeugung,Einfuhr,Ausfuhr


In [19]:
# Check for duplicate product-year combinations
duplicates = quota_results.duplicated(
    subset=["Wirtschaftsjahr", "Bilanz Produkt"],
    keep=False
)

# Display the number of rows involved in duplicates
print("Duplicate product-year rows:", duplicates.sum())

# Display duplicate observations, if any
display(quota_results.loc[duplicates])

Duplicate product-year rows: 0


,Wirtschaftsjahr,Bilanz Produkt,Importquote,Exportquote


In [20]:
import pandas as pd

# Load the restored standardized domestic use dataset
file_path = r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx"

df = pd.read_excel(file_path)

# Display the first five rows
display(df.head())

# Check the structure and data types
df.info()

# Display all column names
print(df.columns.tolist())

,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.0
1,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert,1 000 t Getreidewert,10.0
2,1991/92,Gerste,Detailposition,Industrielle Verwertung für Energie,1 000 t Produktgewicht,NaN
3,1991/92,Gerste,Detailposition,Saatgut,1 000 t Produktgewicht,388.0
4,1991/92,Gerste,Detailposition,Industrielle Verwertung,1 000 t Produktgewicht,2671.0


<class 'pandas.DataFrame'>
RangeIndex: 1734 entries, 0 to 1733
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Wirtschaftsjahr     1734 non-null   str    
 1   Bilanz Produkt      1734 non-null   str    
 2   Bilanz Oberbegriff  1734 non-null   str    
 3   Bilanz Position     1734 non-null   str    
 4   Bilanz Einheit      1734 non-null   str    
 5   Bilanz Wert         1670 non-null   float64
dtypes: float64(1), str(5)
memory usage: 81.4 KB
['Wirtschaftsjahr', 'Bilanz Produkt', 'Bilanz Oberbegriff', 'Bilanz Position', 'Bilanz Einheit', 'Bilanz Wert']


In [21]:
# Check the balance positions available in the dataset
print("Available balance positions:")
print(sorted(df["Bilanz Position"].unique()))

# Count rows for the standardized domestic use indicator
standardized_count = (
    df["Bilanz Position"]
    == "Inlandsverwendung standardisiert"
).sum()

print(
    "\nRows for standardized domestic use:",
    standardized_count
)

# Display the corresponding observations
display(
    df.loc[
        df["Bilanz Position"] == "Inlandsverwendung standardisiert"
    ].head(10)
)

Available balance positions:
['Anfangsbestand', 'Ausfuhr', 'Bestandsänderung', 'Einfuhr', 'Endbestand', 'Futter', 'Gesamterzeugung', 'Industrielle Verwertung', 'Industrielle Verwertung für Energie', 'Inlandsverwendung', 'Inlandsverwendung insgesamt', 'Inlandsverwendung standardisiert', 'Nahrungsverbrauch', 'Nahrungsverbrauch Ausbeute in %', 'Nahrungsverbrauch Kartoffelerzeugnisse', 'Nahrungsverbrauch in Getreidewert', 'Nahrungsverbrauch in Getreidewert in kg je Kopf', 'Nahrungsverbrauch in Produktgewicht', 'Nahrungsverbrauch in Produktgewicht in kg je Kopf', 'Nahrungsverbrauch in Produktgewicht in kg je Kopf/ dgl. kg je Kopf', 'Nahrungsverbrauch in Produktgewicht/ in Mehlwert', 'Nahrungsverbrauch in kg je Kopf', 'Saatgut', 'Selbstversorgungsgrad in %', 'Verluste', 'Verwendbare Erzeugung']

Rows for standardized domestic use: 102


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
1632,1991/92,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10637.0
1633,1991/92,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10787.0
1634,1991/92,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11639.0
1635,1992/93,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10502.0
1636,1992/93,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10644.0
1637,1992/93,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11004.0
1638,1993/94,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,9692.0
1639,1993/94,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11912.0
1640,1993/94,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,12943.0
1641,1994/95,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,9901.0


In [23]:
# 1. Define the output path with a new filename
output_path = r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_1991-2025.xlsx"

# 2. Select the quota columns
# Keep exactly one quota record per financial year and product
quota_results_unique = quota_results[
    ["Wirtschaftsjahr", "Bilanz Produkt", "Importquote", "Exportquote"]
].drop_duplicates(
    subset=["Wirtschaftsjahr", "Bilanz Produkt"]
)

# 3. Create a copy of the original dataset
# The DataFrame df remains unchanged
df_with_quotas = df.copy()

# 4. Add the quota columns using financial year and product as keys
# Each quota is assigned to all balance positions of the corresponding product and year
df_with_quotas = df_with_quotas.merge(
    quota_results_unique,
    on=["Wirtschaftsjahr", "Bilanz Produkt"],
    how="left",
    validate="many_to_one"
)

# 5. Save the expanded dataset under the new filename
df_with_quotas.to_excel(output_path, index=False)

# 6. Verify the result
print(f"File saved: {output_path}")
print("Original number of rows:", len(df))
print("Number of rows after adding quotas:", len(df_with_quotas))
print("Missing import quotas:", df_with_quotas["Importquote"].isna().sum())
print("Missing export quotas:", df_with_quotas["Exportquote"].isna().sum())

# 7. Display the updated dataset
display(df_with_quotas.head())

File saved: C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_with_quotas_1991-2025.xlsx
Original number of rows: 1734
Number of rows after adding quotas: 1734
Missing import quotas: 0
Missing export quotas: 0


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert,Importquote,Exportquote
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.0,10.114848,29.015587
1,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert,1 000 t Getreidewert,10.0,10.114848,29.015587
2,1991/92,Gerste,Detailposition,Industrielle Verwertung für Energie,1 000 t Produktgewicht,NaN,10.114848,29.015587
3,1991/92,Gerste,Detailposition,Saatgut,1 000 t Produktgewicht,388.0,10.114848,29.015587
4,1991/92,Gerste,Detailposition,Industrielle Verwertung,1 000 t Produktgewicht,2671.0,10.114848,29.015587
